# Task 1 — OOF prediction disagreement

Read-only comparison of saved out-of-fold predictions. No fitting and no Test labels.


In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.pipeline.random_seed import set_seed
SEED = set_seed()
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import fbeta_score, mean_absolute_error
SRC = ROOT / "src"


## Inventory and comparable scores
Scores are mean fold macro F₂; repeated folds are not independent.


In [2]:
TASK = SRC / "task1"
RES = TASK / "models" / "results"
card = json.loads((TASK / "data" / "eda" / "eda_original_v1.json").read_text(encoding="utf-8"))
train = pd.read_parquet(SRC / card["splits"]["train"]["path"])
files = {
 "pruned": "round4_lgbm_combined_pruned_oof_v1",
 "original": "round4_lgbm_original_oof_v1",
 "behavior": "t1_lgbm_behavior_threshold_cv6_v3-oof",
 "shallow": "t1_lgbm_shallower_more_trees_threshold_cv6_v3-oof",
 "fixed_behavior": "t1_lgbm_behavior_fixed040_cv6_v3-oof",
 "fixed_shallow": "t1_lgbm_shallower_more_trees_fixed040_cv6_v3-oof",
 "fixed_all": "t1_lgbm_all_fixed040_cv6_v3-oof",
 "fixed_context": "t1_lgbm_context_fixed040_cv6_v3-oof",
 "fixed_more_trees": "t1_lgbm_more_trees_fixed040_cv6_v3-oof",
 "fixed_small_leaf": "t1_lgbm_small_leaf_fixed040_cv6_v3-oof",
 "fixed_balanced": "t1_lgbm_balanced_fixed040_cv6_v4-oof",
}
rows = []
for repeat in (1, 2):
 for name, stem in files.items():
  if name in ("pruned", "original"):
   stem = stem if repeat == 1 else stem.replace("_oof_v1", "_repeat2_oof_v1")
  d = pd.read_parquet(RES / f"{stem}.parquet")
  d = d.rename(columns={"pred_fixed": "pred"})
  if name not in ("pruned", "original"):
   assert len(d) == 2*len(train)
   d = d.loc[(d.fold <= 3) if repeat == 1 else (d.fold > 3)].copy()
  assert d.row_index.is_unique and len(d) == len(train)
  assert np.array_equal(d.sort_values("row_index").true, train.is_cheater)
  assert np.array_equal(d.pred.astype(int), d.pred)
  d = d[["row_index", "fold", "true", "pred", "prob"]].copy()
  d["model"] = name; d["repeat"] = repeat
  rows.append(d)
oof = pd.concat(rows, ignore_index=True)
oof["wrong"] = oof.pred != oof.true
score = (oof.groupby(["model", "fold"], sort=True)
 .apply(lambda g: fbeta_score(g.true, g.pred, beta=2, average="macro", zero_division=0), include_groups=False)
 .rename("macro_F2").reset_index())
ranking = score.groupby("model").agg(mean_macro_F2=("macro_F2", "mean"), folds=("fold", "size"))
ranking["predicted_cheaters_pct"] = oof.groupby("model").pred.mean().mul(100)
ranking["error_pct"] = oof.groupby("model").wrong.mean().mul(100)
display(ranking.sort_values("mean_macro_F2", ascending=False).round(4))
print("Same 90,816 accounts recur in two repeats; fixed-threshold and inner-CV threshold procedures are separate.")


,mean_macro_F2,folds,predicted_cheaters_pct,error_pct
model,,,,
fixed_shallow,0.7778,6,38.5037,21.0002
shallow,0.7775,6,41.2730,21.6025
pruned,0.7773,6,40.2969,21.4180
fixed_behavior,0.7773,6,38.6243,21.0767
behavior,0.7769,6,40.7114,21.5408
original,0.7768,6,40.2313,21.4560
fixed_small_leaf,0.7766,6,38.6622,21.1444
fixed_context,0.7765,6,38.6672,21.1593
fixed_all,0.7763,6,38.6540,21.1703


Same 90,816 accounts recur in two repeats; fixed-threshold and inner-CV threshold procedures are separate.


## Pairwise prediction changes
The first repeat is used for counts so accounts are counted once.


In [3]:
def pair(a, b, repeat=1):
 x = oof.query("model == @a and repeat == @repeat").set_index("row_index").sort_index()
 y = oof.query("model == @b and repeat == @repeat").set_index("row_index").sort_index()
 assert x.index.equals(y.index) and x.true.equals(y.true) and x.fold.equals(y.fold)
 z = pd.DataFrame({"true": x.true, "a": x.pred, "b": y.pred,
                   "a_wrong": x.wrong, "b_wrong": y.wrong, "a_prob": x.prob, "b_prob": y.prob})
 z["changed"] = z.a != z.b
 z["a_only_wrong"] = z.a_wrong & ~z.b_wrong
 z["b_only_wrong"] = z.b_wrong & ~z.a_wrong
 return z

comparisons = [("pruned", "original"), ("pruned", "shallow"), ("pruned", "behavior"),
               ("fixed_behavior", "fixed_shallow"), ("fixed_behavior", "fixed_all"),
               ("fixed_behavior", "fixed_balanced")]
out = []
for a, b in comparisons:
 z = pair(a, b)
 out.append({"a": a, "b": b, "score_a": ranking.loc[a, "mean_macro_F2"],
             "score_b": ranking.loc[b, "mean_macro_F2"], "changed": int(z.changed.sum()),
             "changed_pct": 100*z.changed.mean(), "a_only_wrong": int(z.a_only_wrong.sum()),
             "b_only_wrong": int(z.b_only_wrong.sum()),
             "a_cheaters": int(z.a.sum()), "b_cheaters": int(z.b.sum())})
display(pd.DataFrame(out).round(4))
for repeat in (1, 2):
 z = pair("pruned", "original", repeat)
 print("pruned vs original, repeat", repeat, "changed", int(z.changed.sum()),
       "rescued by original", int(z.a_only_wrong.sum()), "broken by original", int(z.b_only_wrong.sum()))


,a,b,score_a,score_b,changed,changed_pct,a_only_wrong,b_only_wrong,a_cheaters,b_cheaters
0,pruned,original,0.7773,0.7768,4707,5.1830,2126,2581,36101,37948
1,pruned,shallow,0.7773,0.7775,4395,4.8395,2002,2393,36101,38000
2,pruned,behavior,0.7773,0.7769,3856,4.2459,1791,2065,36101,36977
3,fixed_behavior,fixed_shallow,0.7773,0.7778,4191,4.6148,2178,2013,35093,34978
4,fixed_behavior,fixed_all,0.7773,0.7763,3601,3.9652,1760,1841,35093,35142
5,fixed_behavior,fixed_balanced,0.7773,0.7597,13614,14.9908,4893,8721,35093,48687


pruned vs original, repeat 1 changed 4707 rescued by original 2126 broken by original 2581
pruned vs original, repeat 2 changed 4908 rescued by original 2647 broken by original 2261


## Distribution of disagreements
Quartile cuts are descriptive and derived from labeled Train only.


In [4]:
z = pair("pruned", "original", 1).join(train.drop(columns="is_cheater"), how="left")
z["winner"] = np.select([z.a_only_wrong, z.b_only_wrong], ["original", "pruned"], default="same")
for feature in ["reports_received", "friend_network_size", "account_age_days", "crosshair_placement", "level_progression_speed"]:
 bucket = pd.qcut(z[feature], 4, duplicates="drop").astype(str).replace("nan", "missing")
 part = z.assign(bucket=bucket).groupby("bucket", dropna=False).agg(
  rows=("changed", "size"), changed=("changed", "sum"),
  original_rescues=("a_only_wrong", "sum"), original_breaks=("b_only_wrong", "sum"))
 part["changed_pct"] = part.changed / part.rows * 100
 print("\n", feature); display(part.round(2))
display(pd.crosstab(z.loc[z.changed, "true"], z.loc[z.changed, "winner"], rownames=["true class"]))



 reports_received


,rows,changed,original_rescues,original_breaks,changed_pct
bucket,,,,,
"(12.0, 16.0]",17746,1004,479,525,5.66
"(16.0, 29.0]",16967,181,89,92,1.07
"(3.999, 9.0]",21139,1285,585,700,6.08
"(9.0, 12.0]",14722,904,402,502,6.14
missing,20242,1333,571,762,6.59



 friend_network_size


,rows,changed,original_rescues,original_breaks,changed_pct
bucket,,,,,
"(-0.001, 64.0]",17756,366,153,213,2.06
"(135.0, 206.0]",17741,1240,552,688,6.99
"(206.0, 349.0]",17509,834,383,451,4.76
"(64.0, 135.0]",17570,1087,511,576,6.19
missing,20240,1180,527,653,5.83



 account_age_days


,rows,changed,original_rescues,original_breaks,changed_pct
bucket,,,,,
"(1.004, 156.359]",19419,366,159,207,1.88
"(156.359, 479.54]",19419,1355,620,735,6.98
"(479.54, 762.872]",19418,1384,630,754,7.13
"(762.872, 1499.65]",19419,798,365,433,4.11
missing,13141,804,352,452,6.12



 crosshair_placement


,rows,changed,original_rescues,original_breaks,changed_pct
bucket,,,,,
"(0.799, 0.848]",19797,958,436,522,4.84
"(0.848, 0.883]",19797,1180,511,669,5.96
"(0.883, 0.918]",19796,1148,529,619,5.80
"(0.918, 0.96]",19797,770,340,430,3.89
missing,11629,651,310,341,5.60



 level_progression_speed


,rows,changed,original_rescues,original_breaks,changed_pct
bucket,,,,,
"(0.0019299999999999999, 0.0407]",20860,1120,502,618,5.37
"(0.0407, 0.0813]",20859,1380,595,785,6.62
"(0.0813, 0.214]",20859,1285,620,665,6.16
"(0.214, 80.029]",20859,579,261,318,2.78
missing,7379,343,148,195,4.65


winner,original,pruned
true class,,
0,842,1993
1,1284,588


## Assessment

- **Medium priority:** original-feature and shallower LightGBM are useful diversity backups. Pruned/original change 4,707 of 90,816 labels in the first repeat, despite a mean six-fold macro F₂ gap around 0.0006. Their rescue/break balance reverses in repeat 2.
- **Low priority:** class-balanced fixed-threshold LightGBM changes many labels but loses about 0.0176 macro F₂ versus the matched unweighted fixed-threshold procedure.
- The larger disagreement groups have missing reports and mid-to-high friend counts. This is an association, not evidence of leakage or a winning new feature.
